## Parte 1 — Softmax con temperatura

La función se implementa de forma numéricamente estable restando el máximo de los logits antes de exponenciar: con m = max_j z_j, se calcula p_i = exp((z_i − m)/T) / Σ_j exp((z_j − m)/T), de modo que todos los exponentes son ≤ 0 y no hay riesgo de desbordamiento. Con los logits dados (t0..t5 = 2.0, 1.0, 0.5, 0.2, −1.0, −3.0) y la entropía H = −Σ p_i log2 p_i, los resultados son:

| Temperatura | t0 | t1 | t2 | t3 | t4 | t5 | Entropía (bits) |
|---|---|---|---|---|---|---|---|
| T=0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 | 0.9147 |
| T=1.0 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 | 1.7998 |
| T=2.0 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 | 2.2888 |

Las tres distribuciones suman 1.0. Bajar la temperatura afila la distribución: al pasar de T=1 a T=0.5 la masa de t0 sube de 0.5516 a 0.8231 y la de t5 cae de 0.0037 a 3.7367e-05 (que aparece como 0.0000 a cuatro decimales). Subirla la aplana: con T=2, t0 baja a 0.3583 y t5 sube a 0.0294. La entropía crece monótonamente con T: 0.9147 < 1.7998 < 2.2888 bits.

In [1]:
# T1: código aprobado por el crítico
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def softmax_con_temperatura(z, T):
    """Softmax con temperatura, numéricamente estable: resta el máximo antes de exponenciar."""
    z = np.asarray(z, dtype=float)
    x = z / T
    x = x - np.max(x)  # estabilidad numérica
    e = np.exp(x)
    return e / e.sum()


tokens = [f"t{i}" for i in range(6)]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])
temperaturas = [0.5, 1.0, 2.0]

distribuciones = [softmax_con_temperatura(logits, T) for T in temperaturas]
entropias_bits = [float(-(p * np.log2(p)).sum()) for p in distribuciones]
sumas = [float(p.sum()) for p in distribuciones]

# Validaciones del criterio de éxito
assert all(abs(s - 1.0) < 1e-12 for s in sumas), "Cada distribución debe sumar 1"
assert entropias_bits[0] < entropias_bits[1] < entropias_bits[2], \
    "La entropía debe crecer monotónicamente con T"

# Tabla con cuatro decimales
filas = []
for T, p, h in zip(temperaturas, distribuciones, entropias_bits):
    filas.append([f"T={T}"] + [f"{v:.4f}" for v in p] + [f"{h:.4f}"])
columnas = ["Temperatura"] + tokens + ["Entropía (bits)"]

resultados = {
    "tokens": tokens,
    "logits": logits.tolist(),
    "temperaturas": temperaturas,
    "distribuciones": {f"T={T}": p.tolist() for T, p in zip(temperaturas, distribuciones)},
    "entropias_bits": {f"T={T}": h for T, h in zip(temperaturas, entropias_bits)},
    "sumas_distribuciones": {f"T={T}": s for T, s in zip(temperaturas, sumas)},
    "tabla_4_decimales": {"columnas": columnas, "filas": filas},
}
with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# Figura: tabla en PNG
fig, ax = plt.subplots(figsize=(9.5, 2.8))
ax.axis("off")
tbl = ax.table(cellText=filas, colLabels=columnas, loc="center", cellLoc="center")
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1, 1.7)
ax.set_title("Softmax con temperatura: distribuciones y entropía (bits)", fontsize=11)
plt.savefig("tabla_softmax_temperatura.png", dpi=120, bbox_inches="tight")
plt.close(fig)

print("Distribuciones (sin redondear):")
for T, p in zip(temperaturas, distribuciones):
    print(f"  T={T}: {np.round(p, 6).tolist()}")
print("Entropías (bits):", {f"T={T}": round(h, 4) for T, h in zip(temperaturas, entropias_bits)})
print("Sumas:", sumas)
print("Monotonía H(0.5) < H(1) < H(2):",
      entropias_bits[0] < entropias_bits[1] < entropias_bits[2])
print("Archivos: resultados.json, tabla_softmax_temperatura.png")


Distribuciones (sin redondear):
  T=0.5: [0.823065, 0.11139, 0.040978, 0.022489, 0.00204, 3.7e-05]
  T=1.0: [0.551623, 0.202931, 0.123084, 0.091183, 0.027464, 0.003717]
  T=2.0: [0.358335, 0.217341, 0.169266, 0.145688, 0.079955, 0.029414]
Entropías (bits): {'T=0.5': 0.9147, 'T=1.0': 1.7998, 'T=2.0': 2.2888}
Sumas: [1.0, 1.0, 1.0]
Monotonía H(0.5) < H(1) < H(2): True
Archivos: resultados.json, tabla_softmax_temperatura.png


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T=1, los tokens ya están en orden descendente de probabilidad (t0, t1, t2, t3, t4, t5) y las probabilidades acumuladas son 0.5516, 0.7546, 0.8776, 0.9688, 0.9963 y 1.0000. Con p = 0.9, el conjunto {t0, t1, t2} acumula solo 0.8776 (< 0.9), por lo que el núcleo mínimo que alcanza el umbral es {t0, t1, t2, t3}, con probabilidad acumulada 0.9688. Los tokens t4 y t5 quedan fuera del núcleo. Renormalizando (dividiendo cada probabilidad entre 0.9688):

| Token | Prob. T=1 | Acumulada | En núcleo | Prob. renormalizada |
|---|---|---|---|---|
| t0 | 0.5516 | 0.5516 | sí | 0.5694 |
| t1 | 0.2029 | 0.7546 | sí | 0.2095 |
| t2 | 0.1231 | 0.8776 | sí | 0.1270 |
| t3 | 0.0912 | 0.9688 | sí | 0.0941 |
| t4 | 0.0275 | 0.9963 | no | 0.0000 |
| t5 | 0.0037 | 1.0000 | no | 0.0000 |

Sobreviven 4 tokens (t0, t1, t2, t3); la distribución renormalizada suma 1.0 y asigna 0.0000 a t4 y t5.

In [2]:
# T2: código aprobado por el crítico
import json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

p = 0.9
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])

# Distribución T=1 obtenida en la Parte 1 (entrada/T1/resultados.json)
try:
    with open("entrada/T1/resultados.json", encoding="utf-8") as f:
        prev = json.load(f)
    tokens = list(prev["tokens"])
    probs = np.array(prev["distribuciones"]["T=1.0"], dtype=float)
except FileNotFoundError:
    z = logits - logits.max()
    probs = np.exp(z) / np.exp(z).sum()

assert abs(probs.sum() - 1.0) < 1e-9 and probs.min() >= 0

# Orden descendente por probabilidad y acumuladas
orden = np.argsort(-probs, kind="stable")
probs_ord = probs[orden]
tokens_ord = [tokens[i] for i in orden]
acum = np.cumsum(probs_ord)

# Conjunto mínimo cuya probabilidad acumulada alcanza p = 0.9
k = min(int(np.searchsorted(acum, p, side="left")) + 1, len(tokens))
nucleo_idx = orden[:k]
nucleo_tokens = tokens_ord[:k]
acum_nucleo = float(acum[k - 1])
fuera = [t for t in tokens if t not in nucleo_tokens]

# Renormalización del núcleo; tokens fuera del núcleo quedan en 0
renorm_nucleo = probs_ord[:k] / acum_nucleo
renorm = np.zeros(len(tokens))
renorm[nucleo_idx] = renorm_nucleo

# Validaciones
assert acum_nucleo >= p, "la acumulada del núcleo debe alcanzar p"
assert k == 1 or acum[k - 2] < p, "el núcleo debe ser el conjunto mínimo"
assert abs(renorm.sum() - 1.0) < 1e-12
assert np.all(renorm >= 0.0)

# Figura: distribución T=1 vs renormalizada tras top-p
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(tokens))
w = 0.38
ax.bar(x - w / 2, probs, w, label="T=1 original", color="#4C72B0")
ax.bar(x + w / 2, renorm, w, label="Top-p (p=0.9) renormalizada", color="#DD8452")
for i in range(len(tokens)):
    ax.text(x[i] - w / 2, probs[i] + 0.008, f"{probs[i]:.4f}", ha="center", fontsize=8)
    v = renorm[i]
    ax.text(x[i] + w / 2, v + 0.008, f"{v:.4f}" if v > 0 else "0", ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_ylim(0, 0.75)
ax.set_ylabel("Probabilidad")
ax.set_title(f"Top-p (núcleo) con p={p} sobre la distribución T=1\n"
             f"Núcleo: {', '.join(nucleo_tokens)} (acumulada {acum_nucleo:.4f})")
ax.legend()
fig.tight_layout()
fig.savefig("top_p_renormalizada.png", dpi=120)
plt.close(fig)

tabla_filas = [
    [tokens_ord[i], float(probs_ord[i]), float(acum[i]), bool(i < k),
     float(renorm_nucleo[i]) if i < k else 0.0]
    for i in range(len(tokens))
]

resultados = {
    "p": p,
    "temperatura": 1.0,
    "tokens": tokens,
    "distribucion_T1": probs.tolist(),
    "orden_descendente": tokens_ord,
    "probabilidades_ordenadas": probs_ord.tolist(),
    "probabilidades_acumuladas": acum.tolist(),
    "tokens_nucleo": nucleo_tokens,
    "num_tokens_nucleo": k,
    "probabilidad_acumulada_nucleo": acum_nucleo,
    "tokens_fuera_nucleo": fuera,
    "distribucion_renormalizada": {t: float(v) for t, v in zip(tokens, renorm)},
    "distribucion_renormalizada_nucleo": {t: float(v) for t, v in zip(nucleo_tokens, renorm_nucleo)},
    "suma_distribucion_renormalizada": float(renorm.sum()),
    "tabla": {
        "columnas": ["Token", "Prob. T=1", "Acumulada", "En_nucleo", "Prob. renormalizada"],
        "filas": tabla_filas,
    },
}

with open("resultados.json", "w", encoding="utf-8") as f:
    json.dump(resultados, f, indent=2, ensure_ascii=False)

print("Distribución T=1:", dict(zip(tokens, np.round(probs, 6))))
print("Orden descendente:", tokens_ord)
print("Acumuladas:", np.round(acum, 6).tolist())
print(f"Núcleo top-p (p={p}): {nucleo_tokens} (mínimo, acumulada={acum_nucleo:.6f} >= 0.9)")
print("Fuera del núcleo (probabilidad 0):", fuera)
print("Distribución renormalizada:", {t: round(float(v), 6) for t, v in zip(tokens, renorm)})
print("Suma renormalizada:", float(renorm.sum()))
print("Figura guardada: top_p_renormalizada.png")


Distribución T=1: {'t0': np.float64(0.551623), 't1': np.float64(0.202931), 't2': np.float64(0.123084), 't3': np.float64(0.091183), 't4': np.float64(0.027464), 't5': np.float64(0.003717)}
Orden descendente: ['t0', 't1', 't2', 't3', 't4', 't5']
Acumuladas: [0.551623, 0.754553, 0.877637, 0.96882, 0.996283, 1.0]
Núcleo top-p (p=0.9): ['t0', 't1', 't2', 't3'] (mínimo, acumulada=0.968820 >= 0.9)
Fuera del núcleo (probabilidad 0): ['t4', 't5']
Distribución renormalizada: {'t0': 0.569376, 't1': 0.209462, 't2': 0.127045, 't3': 0.094117, 't4': 0.0, 't5': 0.0}
Suma renormalizada: 1.0
Figura guardada: top_p_renormalizada.png


## Parte 3 — Comprobación empírica

Se tomaron 10000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). Los conteos y frecuencias observadas, comparados con las probabilidades teóricas, son:

| Token | Prob. teórica | Conteo observado | Frecuencia observada | Diferencia |
|---|---|---|---|---|
| t0 | 0.5694 | 5682 | 0.5682 | -0.0012 |
| t1 | 0.2095 | 2093 | 0.2093 | -0.0002 |
| t2 | 0.1270 | 1296 | 0.1296 | 0.0026 |
| t3 | 0.0941 | 929 | 0.0929 | -0.0012 |
| t4 | 0.0000 | 0 | 0.0000 | 0.0000 |
| t5 | 0.0000 | 0 | 0.0000 | 0.0000 |

La divergencia KL de la distribución empírica respecto de la teórica es 5.0069e-05 bits (3.4705e-05 nats), y la mayor desviación absoluta entre frecuencia observada y probabilidad teórica es 0.0026, en t2. El gráfico de barras generado en la celda muestra pares de barras (observada vs. teórica) prácticamente indistinguibles: las diferencias son del orden del ruido estadístico esperable con 10000 muestras, lo que confirma empíricamente que el muestreo sigue la distribución renormalizada del núcleo.

In [3]:
# T3: código aprobado por el crítico
import json
from pathlib import Path
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# --- Distribución renormalizada de la Parte 2 (resultado previo T2) ---
with open("entrada/T2/resultados.json", encoding="utf-8") as f:
    t2 = json.load(f)

tokens = t2["tokens"]
p_teo = np.array([t2["distribucion_renormalizada"][t] for t in tokens], dtype=float)

# Validación: recalcular top-p (T=1, p=0.9) desde los logits y comparar
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])
z = logits - logits.max()
p1 = np.exp(z) / np.exp(z).sum()
orden = np.argsort(-p1)
k = int(np.searchsorted(np.cumsum(p1[orden]), 0.9, side="left") + 1)
p_check = np.zeros_like(p1)
nuc = orden[:k]
p_check[nuc] = p1[nuc] / p1[nuc].sum()
assert np.allclose(p_check, p_teo, atol=1e-9), "Inconsistencia con la distribución renormalizada de T2"

# --- Muestreo: 10,000 muestras con numpy.random.default_rng(0) ---
N = 10000
rng = np.random.default_rng(0)
muestras = rng.choice(len(tokens), size=N, p=p_teo)
conteos = np.bincount(muestras, minlength=len(tokens))
freq_obs = conteos / N

# --- Divergencia KL (bits) de la empírica respecto de la teórica ---
# Términos con frecuencia observada 0 aportan 0; no hay p_teo=0 con freq>0 (se muestreó de p_teo)
mask = freq_obs > 0
kl_bits = float(np.sum(freq_obs[mask] * np.log2(freq_obs[mask] / p_teo[mask])))
kl_nats = float(np.sum(freq_obs[mask] * np.log(freq_obs[mask] / p_teo[mask])))

# --- Gráfico de barras: frecuencias observadas vs probabilidades teóricas ---
x = np.arange(len(tokens))
w = 0.38
fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(x - w / 2, freq_obs, w, label="Frecuencia observada", color="#4C72B0")
ax.bar(x + w / 2, p_teo, w, label="Probabilidad teórica", color="#DD8452")
ax.set_xticks(x)
ax.set_xticklabels(tokens)
ax.set_xlabel("Token")
ax.set_ylabel("Frecuencia / probabilidad")
ax.set_title(f"Parte 3: muestreo top-p (T=1, p=0.9), N={N}, semilla=0\n"
             f"KL(empírica || teórica) = {kl_bits:.6f} bits")
ax.legend()
for xi, (fo, pt) in enumerate(zip(freq_obs, p_teo)):
    ax.text(xi - w / 2, fo, f"{fo:.4f}", ha="center", va="bottom", fontsize=8)
    if pt > 0:
        ax.text(xi + w / 2, pt, f"{pt:.4f}", ha="center", va="bottom", fontsize=8)
fig.tight_layout()
fig.savefig("parte3_frecuencias_vs_teoricas.png", dpi=120)
plt.close(fig)

# --- resultados.json (contrato de la subtarea) ---
resultados = {
    "subtarea": "T3",
    "num_muestras": N,
    "semilla": 0,
    "tokens": tokens,
    "probabilidades_teoricas": p_teo.tolist(),
    "conteos_observados": conteos.tolist(),
    "frecuencias_observadas": freq_obs.tolist(),
    "divergencia_KL_bits": kl_bits,
    "divergencia_KL_nats": kl_nats,
    "desviacion_maxima_abs": float(np.max(np.abs(freq_obs - p_teo))),
    "figura": "parte3_frecuencias_vs_teoricas.png",
    "tabla": {
        "columnas": ["Token", "Prob. teórica", "Conteo observado", "Frecuencia observada", "Diferencia"],
        "filas": [[t, float(pt), int(c), float(fo), float(fo - pt)]
                  for t, pt, c, fo in zip(tokens, p_teo, conteos, freq_obs)],
    },
}
Path("resultados.json").write_text(json.dumps(resultados, indent=2, ensure_ascii=False), encoding="utf-8")

# --- Resumen ---
print("T3 — Comprobación empírica (N=10000, semilla=0)")
for t, pt, c, fo in zip(tokens, p_teo, conteos, freq_obs):
    print(f"  {t}: teórica={pt:.6f}  observada={fo:.6f}  (conteo={c})")
print(f"Divergencia KL (bits): {kl_bits:.8f}")
print(f"Divergencia KL (nats): {kl_nats:.8f}")
print("Figura guardada: parte3_frecuencias_vs_teoricas.png")


T3 — Comprobación empírica (N=10000, semilla=0)
  t0: teórica=0.569376  observada=0.568200  (conteo=5682)
  t1: teórica=0.209462  observada=0.209300  (conteo=2093)
  t2: teórica=0.127045  observada=0.129600  (conteo=1296)
  t3: teórica=0.094117  observada=0.092900  (conteo=929)
  t4: teórica=0.000000  observada=0.000000  (conteo=0)
  t5: teórica=0.000000  observada=0.000000  (conteo=0)
Divergencia KL (bits): 0.00005007
Divergencia KL (nats): 0.00003471
Figura guardada: parte3_frecuencias_vs_teoricas.png


## Parte 4 — Pregunta conceptual

Cuando T → 0, dividir los logits por una temperatura cada vez más pequeña amplifica sus diferencias: el término exp(z_max/T) domina exponencialmente al resto y toda la masa se concentra en el token de mayor logit, aquí t0. La distribución tiende a una masa puntual y su entropía tiende a 0. Las entropías de la Parte 1 evidencian esa tendencia: al reducir T de 1 a 0.5 la entropía ya cae de 1.7998 a 0.9147 bits, la probabilidad de t0 sube de 0.5516 a 0.8231 y la de t5 se desploma hasta 3.7367e-05. Por eso T → 0 equivale a la decodificación voraz: greedy elige en cada paso argmax(z), y el límite T → 0 del softmax asigna probabilidad 1 exactamente a ese mismo token, de modo que muestrear de esa distribución determinística produce siempre la misma elección que greedy.

Cuando T → ∞, z_i/T → 0 para todo token, todos los exponenciales valen lo mismo y p_i → 1/6: la distribución tiende a la uniforme sobre el vocabulario y la entropía tiende a su máximo posible, log2 6. La evidencia de la Parte 1 apunta en esa dirección: al duplicar T de 1 a 2 la entropía sube de 1.7998 a 2.2888 bits y las probabilidades se aplanan (t0 pasa de 0.5516 a 0.3583; t5 de 0.0037 a 0.0294). La secuencia monótona 0.9147 → 1.7998 → 2.2888 confirma que la entropía crece con T, desde el régimen casi determinista (T → 0, equivalente a greedy) hasta el régimen casi uniforme (T → ∞).